# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ziadhamouda370-beep/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
I will use a Random Forest classifier. It fits this task because the target is binary and the available search, engagement, freshness, and content signals can interact in nonlinear ways. A Random Forest also provides feature importance that can help explain which signals the model relies on. I will compare its ranking performance against the transparent baseline using the same held-out data.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    recall_score,
    precision_score,
    roc_auc_score
)

url = "https://raw.githubusercontent.com/ziadhamouda370-beep/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

# Binary target
df["is_declining"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Rows:", len(df))
print("Declining:", df["is_declining"].sum())
print("Non-declining:", (df["is_declining"] == 0).sum())
print("Method: Random Forest Classifier")

Rows: 30000
Declining: 16262
Non-declining: 13738
Method: Random Forest Classifier


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*
I will use a client-holdout split. About 20% of unique clients will be held out for testing, while the remaining clients will be used for training. This is more honest than splitting individual rows because multiple content items can belong to the same client. Keeping each client entirely in one split reduces cross-client leakage and better tests whether the model generalizes to unseen clients.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=groups)
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("Train clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

overlap = set(train_df["client_id"]) & set(test_df["client_id"])

print("Client overlap:", len(overlap))

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*
I will train the Random Forest on the training clients only and evaluate it on the held-out clients. I will compare the model with the same transparent baseline on the same test clients. The main comparison will use Precision@50 because this lane produces a ranked review queue: it measures how many of the first 50 recommendations are actually declining items. I will also report recall as a secondary diagnostic.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Safe numeric features: exclude IDs, labels, product flags, and derived target fields
candidate_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate"
]

feature_cols = [
    c for c in candidate_features
    if c in df.columns
]

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train = train_df["is_declining"]
y_test = test_df["is_declining"]

# Fill missing values using training medians only
train_medians = X_train.median(numeric_only=True)

X_train = X_train.fillna(train_medians).fillna(0)
X_test = X_test.fillna(train_medians).fillna(0)

# Train model
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

# Probability of declining
model_probability = model.predict_proba(X_test)[:, 1]

# Rank highest probability first
model_results = test_df[
    ["content_id", "client_id", "is_declining"]
].copy()

model_results["model_score"] = model_probability

model_results = model_results.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

model_results["model_rank"] = np.arange(
    1,
    len(model_results) + 1
)

# Precision@50
k = 50

precision_at_50 = (
    model_results.head(k)["is_declining"].sum() / k
)

# Recall
recall = recall_score(
    y_test,
    (model_probability >= 0.5).astype(int),
    zero_division=0
)

print("Features used:", len(feature_cols))
print("Precision@50:", round(precision_at_50, 3))
print("Recall:", round(recall, 3))

print("\nTop 20 model recommendations:")
display(model_results.head(20))

Features used: 26
Precision@50: 1.0
Recall: 0.928

Top 20 model recommendations:


,content_id,client_id,is_declining,model_score,model_rank
0,content_bf313c86ff6b,client_f369cb89fc,1,0.990000,1
1,content_b23d650634c6,client_f369cb89fc,1,0.990000,2
2,content_0b50482209cd,client_bdd2d3af3a,1,0.990000,3
3,content_eb3b2c3bbc34,client_f369cb89fc,1,0.990000,4
4,content_d028d97a0f72,client_f369cb89fc,1,0.986667,5
5,content_f6bf66378677,client_f369cb89fc,1,0.986667,6
6,content_9234f5075e7a,client_f369cb89fc,1,0.986667,7
7,content_47806094bb4d,client_f369cb89fc,1,0.986667,8
8,content_9bbf99b4dc21,client_434c9b5ae5,1,0.983333,9
9,content_e3bf6539b4ba,client_bdd2d3af3a,1,0.983333,10


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*
I will inspect false positives and false negatives rather than relying only on the aggregate metric. False positives are content items the model ranks as likely declining but whose label is not declining. False negatives are declining items that receive a lower model score. I will also inspect feature importance, while treating it as model-specific evidence rather than causal evidence.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Predictions at the default probability threshold
model_results["predicted_declining"] = (
    model_results["model_score"] >= 0.5
).astype(int)

# False positives
false_positives = model_results[
    (model_results["predicted_declining"] == 1) &
    (model_results["is_declining"] == 0)
].copy()

# False negatives
false_negatives = model_results[
    (model_results["predicted_declining"] == 0) &
    (model_results["is_declining"] == 1)
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nExample false positives:")
display(false_positives.head(10))

print("\nExample false negatives:")
display(false_negatives.head(10))

# Feature importance
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("\nTop model features:")
display(importance.head(10))

False positives: 345
False negatives: 226

Example false positives:


,content_id,client_id,is_declining,model_score,model_rank,predicted_declining
1310,content_816d77e36e14,client_8527a891e2,0,0.803333,1311,1
1323,content_41baf0722ad9,client_8527a891e2,0,0.800000,1324,1
1674,content_4d9f36001f06,client_8527a891e2,0,0.753333,1675,1
1682,content_8f1409b2674e,client_8527a891e2,0,0.750000,1683,1
1801,content_4dd569ee33c9,client_8527a891e2,0,0.733333,1802,1
1829,content_15e0e6081fe9,client_f369cb89fc,0,0.730000,1830,1
1838,content_94fa1e16f2b8,client_4e07408562,0,0.730000,1839,1
1844,content_e0666cdbf9c3,client_4e07408562,0,0.730000,1845,1
1871,content_09227e80fef5,client_8527a891e2,0,0.723333,1872,1
1881,content_5ce1a9d3e4d7,client_8527a891e2,0,0.723333,1882,1



Example false negatives:


,content_id,client_id,is_declining,model_score,model_rank,predicted_declining
3268,content_db3fd75bd4fe,client_f369cb89fc,1,0.496667,3269,0
3269,content_a4d677c4395d,client_e629fa6598,1,0.496667,3270,0
3270,content_557fe9494c53,client_4e07408562,1,0.496667,3271,0
3272,content_1180705d05bb,client_4e07408562,1,0.496667,3273,0
3275,content_2d6c50388f53,client_4e07408562,1,0.496667,3276,0
3276,content_71c7b5968117,client_4e07408562,1,0.496667,3277,0
3278,content_b2cb21940b47,client_f369cb89fc,1,0.496667,3279,0
3281,content_6d772c85e856,client_e629fa6598,1,0.496667,3282,0
3282,content_ec5ed4cd5e41,client_4e07408562,1,0.496667,3283,0
3285,content_b6bb672d24a4,client_4e07408562,1,0.496667,3286,0



Top model features:


,feature,importance
19,impressions_prev_30d,0.217607
16,impressions_last_30d,0.187799
6,impressions_90d,0.083806
4,content_age_days,0.059346
23,avg_position,0.057861
14,days_with_impressions,0.055516
3,word_count,0.032660
18,sessions_last_30d,0.027692
22,ctr,0.024816
17,clicks_last_30d,0.024617


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.